# Notebook 02 — Algoritmo Genético

**Tech Challenge FIAP — Fase 2**

Otimização de hiperparâmetros do Random Forest via Algoritmo Genético.
São realizados **3 experimentos** com configurações diferentes para comparar convergência e resultado.

| Experimento | Pop | Gerações | Mutação | Cruzamento | Seed |
|-------------|-----|----------|---------|------------|------|
| 1 — Padrão       | 30 | 20 | 0.10 | 0.80 | 42 |
| 2 — Alta mutação | 30 | 20 | 0.30 | 0.80 | 43 |
| 3 — Pop. grande  | 60 | 30 | 0.10 | 0.90 | 44 |

**Referência (baseline GridSearch):** Random Forest F1-macro = 0.9017

## 0. Imports e configurações

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib
try:
    get_ipython()  # só existe dentro do Jupyter
except NameError:
    matplotlib.use("Agg")  # backend headless apenas fora do Jupyter

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipelines.preprocessing import build_preprocessed_splits
from src.models.baseline import build_pipelines
from src.genetic_algorithm.ga import run_genetic_algorithm
from src.evaluation.metrics import evaluate_model

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projeto raiz: {PROJECT_ROOT}")

## 1. Dados e pipeline base

In [2]:
X_train, X_test, y_train, y_test, scaler = build_preprocessed_splits()
pipeline_base = build_pipelines()["random_forest"]

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

PIPELINE DE PRE-PROCESSAMENTO
[load] Dataset carregado: 1014 linhas x 7 colunas
[outliers] Removidas 2 linha(s) com HeartRate < 30 bpm.
[age] Removidos 82 registro(s) com Age > 54 anos.
[dedup] Removidas 140 ocorrencia(s) acima do limite de 3 por grupo. Linhas restantes: 790
[features] BodyTemp convertida de Fahrenheit para Celsius.
[split] Treino: 632 linhas | Teste: 158 linhas (20% teste)
[scaler] StandardScaler aplicado (fit no treino, transform em ambos).
-------------------------------------------------------
Features finais (6): ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
Distribuicao do alvo — treino:
  low risk (0): 277 (43.8%)
  mid risk (1): 198 (31.3%)
  high risk (2): 157 (24.8%)
Distribuicao do alvo — teste:
  low risk (0): 69 (43.7%)
  mid risk (1): 50 (31.6%)
  high risk (2): 39 (24.7%)
Treino: (632, 6) | Teste: (158, 6)


## 2. Resultados baseline (referência)

Carregamos os resultados do Notebook 01 para comparação ao final.

In [3]:
with open(EXPERIMENTS_DIR / "baseline_results.json") as f:
    baseline = json.load(f)

baseline_rf = baseline["random_forest"]
print(f"Baseline RF — F1-macro: {baseline_rf['f1_macro']} | ROC-AUC: {baseline_rf['roc_auc_macro_ovr']}")

Baseline RF — F1-macro: 0.9017 | ROC-AUC: 0.9813


## 3. Experimento 1 — Configuração Padrão

`pop=30 | gen=20 | mut=0.10 | cross=0.80`

In [4]:
result_exp1 = run_genetic_algorithm(
    model_name="random_forest",
    pipeline=build_pipelines()["random_forest"],
    X_train=X_train,
    y_train=y_train,
    population_size=30,
    generations=20,
    mutation_rate=0.10,
    crossover_rate=0.80,
    random_state=42,
)

print(f"\nMelhor indivíduo: {result_exp1['best_individual']}")
print(f"Melhor F1-macro CV: {result_exp1['best_fitness']:.4f}")


Algoritmo Genético — random_forest
Pop: 30 | Gerações: 20 | Mutação: 0.1 | Cruzamento: 0.8
[Gen   1/20] best=0.7857 | mean=0.7011 | global_best=0.7857
[Gen   2/20] best=0.7897 | mean=0.7161 | global_best=0.7897
[Gen   3/20] best=0.7897 | mean=0.7327 | global_best=0.7897
[Gen   4/20] best=0.7897 | mean=0.7579 | global_best=0.7897
[Gen   5/20] best=0.7928 | mean=0.7650 | global_best=0.7928
[Gen   6/20] best=0.7928 | mean=0.7728 | global_best=0.7928
[Gen   7/20] best=0.7963 | mean=0.7808 | global_best=0.7963
[Gen   8/20] best=0.7963 | mean=0.7817 | global_best=0.7963
[Gen   9/20] best=0.7963 | mean=0.7538 | global_best=0.7963
[Gen  10/20] best=0.7963 | mean=0.7807 | global_best=0.7963
[Gen  11/20] best=0.7963 | mean=0.7770 | global_best=0.7963
[Gen  12/20] best=0.7963 | mean=0.7835 | global_best=0.7963
[Gen  13/20] best=0.7963 | mean=0.7668 | global_best=0.7963
[Gen  14/20] best=0.7963 | mean=0.7840 | global_best=0.7963
[Gen  15/20] best=0.7963 | mean=0.7796 | global_best=0.7963
[Gen  16

## 4. Experimento 2 — Alta Mutação

`pop=30 | gen=20 | mut=0.30 | cross=0.80`

In [ ]:
result_exp2 = run_genetic_algorithm(
    model_name="random_forest",
    pipeline=build_pipelines()["random_forest"],
    X_train=X_train,
    y_train=y_train,
    population_size=30,
    generations=20,
    mutation_rate=0.30,
    crossover_rate=0.80,
    random_state=43,
)

print(f"\nMelhor indivíduo: {result_exp2['best_individual']}")
print(f"Melhor F1-macro CV: {result_exp2['best_fitness']:.4f}")


Algoritmo Genético — random_forest
Pop: 30 | Gerações: 20 | Mutação: 0.3 | Cruzamento: 0.8
[Gen   1/20] best=0.7714 | mean=0.7085 | global_best=0.7714
[Gen   2/20] best=0.7714 | mean=0.7243 | global_best=0.7714
[Gen   3/20] best=0.7714 | mean=0.7304 | global_best=0.7714
[Gen   4/20] best=0.7714 | mean=0.7363 | global_best=0.7714


## 5. Experimento 3 — População Grande

`pop=60 | gen=30 | mut=0.10 | cross=0.90`

In [ ]:
result_exp3 = run_genetic_algorithm(
    model_name="random_forest",
    pipeline=build_pipelines()["random_forest"],
    X_train=X_train,
    y_train=y_train,
    population_size=60,
    generations=30,
    mutation_rate=0.10,
    crossover_rate=0.90,
    random_state=44,
)

print(f"\nMelhor indivíduo: {result_exp3['best_individual']}")
print(f"Melhor F1-macro CV: {result_exp3['best_fitness']:.4f}")

## 6. Curvas de convergência

In [ ]:
experiments = [
    ("Exp 1 — Padrão",       result_exp1, "steelblue"),
    ("Exp 2 — Alta Mutação",  result_exp2, "darkorange"),
    ("Exp 3 — Pop. Grande",   result_exp3, "seagreen"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, result, color in experiments:
    gens = [h["generation"]   for h in result["history"]]
    best = [h["global_best"]  for h in result["history"]]
    mean = [h["mean_fitness"] for h in result["history"]]

    axes[0].plot(gens, best, label=label, color=color, linewidth=2)
    axes[1].plot(gens, mean, label=label, color=color, linewidth=2, linestyle="--")

for ax in axes:
    ax.axhline(baseline_rf["f1_macro"], color="red", linestyle=":", linewidth=1.5,
               label=f"Baseline GridSearch ({baseline_rf['f1_macro']})")
    ax.set_xlabel("Geração")
    ax.set_ylabel("F1-macro")
    ax.legend(fontsize=8)
    ax.set_ylim(0.5, 1.0)

axes[0].set_title("Melhor fitness global por geração")
axes[1].set_title("Fitness médio da população por geração")

fig.savefig(FIGURES_DIR / "ga_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Avaliação no conjunto de teste — modelos otimizados

In [ ]:
from sklearn.pipeline import Pipeline
from src.genetic_algorithm.encoding import decode_to_sklearn_params

resultados_ag = {}

for label, result in [("exp1_padrao", result_exp1),
                       ("exp2_alta_mutacao", result_exp2),
                       ("exp3_pop_grande", result_exp3)]:

    # Recria pipeline e aplica melhores hiperparâmetros
    pipe = build_pipelines()["random_forest"]
    params = decode_to_sklearn_params("random_forest", result["best_individual"])
    pipe.set_params(**params)
    pipe.fit(X_train, y_train)

    metricas = evaluate_model(
        nome=label,
        pipeline=pipe,
        X_test=X_test,
        y_test=y_test,
        plot_confusion=True,
        save_path=str(FIGURES_DIR / f"ga_confusion_{label}.png"),
    )
    metricas["best_individual"] = result["best_individual"]
    metricas["best_fitness_cv"] = result["best_fitness"]
    resultados_ag[label] = metricas

## 8. Comparativo: Baseline vs. AG

In [ ]:
# Monta tabela comparativa
rows = []

rows.append({
    "modelo": "Baseline (GridSearch)",
    "f1_macro": baseline_rf["f1_macro"],
    "roc_auc": baseline_rf["roc_auc_macro_ovr"],
    "recall_high_risk": baseline_rf["recall_high_risk"],
    "accuracy": baseline_rf["accuracy"],
})

labels_display = {
    "exp1_padrao":       "AG Exp.1 — Padrão",
    "exp2_alta_mutacao": "AG Exp.2 — Alta Mutação",
    "exp3_pop_grande":   "AG Exp.3 — Pop. Grande",
}

for key, m in resultados_ag.items():
    rows.append({
        "modelo": labels_display[key],
        "f1_macro": m["f1_macro"],
        "roc_auc": m["roc_auc_macro_ovr"],
        "recall_high_risk": m["recall_high_risk"],
        "accuracy": m["accuracy"],
    })

df_comp = pd.DataFrame(rows).set_index("modelo")
print("Comparativo Final:")
df_comp

## 9. Salvar resultados dos experimentos

In [ ]:
def save_experiment(path: Path, config: dict, result: dict, metricas: dict) -> None:
    data = {
        "name": config["name"],
        "description": config["description"],
        "config": result["config"],
        "results": {
            "best_individual": {k: str(v) for k, v in result["best_individual"].items()},
            "best_fitness_cv": result["best_fitness"],
            "test_metrics": {k: v for k, v in metricas.items() if k != "best_individual"},
            "elapsed_seconds": result["elapsed_seconds"],
        },
        "history": result["history"],
        "status": "completed",
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Salvo: {path.name}")


configs = [
    {"name": "Experimento 1 — Configuração Padrão",    "description": "Baseline do AG para comparação"},
    {"name": "Experimento 2 — Alta Mutação",            "description": "Taxa de mutação elevada para maior exploração"},
    {"name": "Experimento 3 — População Grande",        "description": "População maior para convergência mais precisa"},
]

for i, (result, key) in enumerate([
    (result_exp1, "exp1_padrao"),
    (result_exp2, "exp2_alta_mutacao"),
    (result_exp3, "exp3_pop_grande"),
]):
    path = EXPERIMENTS_DIR / f"experiment_0{i+1}_{'_'.join(configs[i]['name'].split()[2:]).lower().replace('ã', 'a').replace('ç', 'c')}.json"
    # usa os arquivos já existentes
    existing_files = ["experiment_01_default.json", "experiment_02_high_mutation.json", "experiment_03_large_population.json"]
    save_experiment(EXPERIMENTS_DIR / existing_files[i], configs[i], result, resultados_ag[key])